# Challenge 1: Setup and baseline

You'll evaluate one agent for the whole track: an **order-support agent** with four tools.
Here you connect LangSmith, read the agent's code, and capture a baseline trace.

Run each cell in order: click it, then press `Shift+Enter`.

## 1. Enter your LangSmith key

The key is stored in `.env` at the repo root, which is git-ignored. It is required: every dataset, experiment, and trace in this track lives in your own LangSmith workspace.

In [ ]:
from lab_credentials import collect

collect()

## 2. Meet the agent

`eval_lab/agent.py` defines four tools over in-memory data, so runs are fast and repeatable:

| Tool | Purpose |
|---|---|
| `lookup_order` | Item, status, total, days since delivery |
| `check_refund_eligibility` | Delivered within 30 days? |
| `issue_refund` | Issue a refund (should only run for eligible orders) |
| `search_faq` | Returns, shipping, and warranty policy |

Print the tool names and the system prompt the agent starts with.

In [ ]:
import sys
for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS

print("tools :", [t.name for t in TOOLS])
print("prompt:", SYSTEM_PROMPT_V1)

Read the prompt. It tells the agent to avoid extra steps and to issue refunds after looking up the order. Nothing there says to check eligibility. Hold that thought: the evals in challenges 3 and 4 find out whether it matters.

## 3. Run the agent once

Order `A1002` was delivered 45 days ago, outside the 30-day refund window. A correct agent should decline.

The `run_id` in the config names the trace, so the link at the end opens it directly.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import uuid7
from eval_lab.agent import build_agent
from lab_trace import trace_link

agent = build_agent()
run_id = uuid7()
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Please refund order A1002."}]},
    config={"run_id": run_id},
)

for m in result["messages"]:
    m.pretty_print()

trace_link(run_id)

**Result:** a conversation with tool calls, then a final answer. Open the trace link and expand the run: each tool call is a child span. The answer might look fine, and that is the problem. A final answer alone doesn't tell you *how* the agent got there.

## 4. Verify the connection

Re-checks your key from a fresh process.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from lab_credentials import TRACING_BLOCKED, TRACING_FIX, probe_tracing

blocked = probe_tracing(
    os.environ["LANGSMITH_API_KEY"],
    os.environ.get("LANGSMITH_ENDPOINT") or "https://api.smith.langchain.com",
    os.environ["LANGSMITH_PROJECT"],
)
if blocked:
    raise SystemExit(f"{TRACING_BLOCKED[blocked]} {TRACING_FIX}")
print("✓ LangSmith key works and the workspace accepts tracing")
print("✓ project:", os.environ["LANGSMITH_PROJECT"])

---
Every line above should show ✓.